# 01 · GLM-OCR Anatomy: Config → Visual Tokens

**Hardware**: 🟢 CPU. The structural path uses a miniature randomly initialised model; no model weights are downloaded.

This notebook repeats the method used throughout Part I: read the released JSON, instantiate a tiny model with the same topology, hook the real library symbols, and verify one implementation detail numerically.

> Version target: `transformers` 5.14.x. GLM-OCR entered the native Transformers implementation after the checkpoint's first release; older versions may not expose these classes.

## 0. Setup

Install into a clean environment if needed. Keeping the version aligned with the book prevents source lines and class defaults from drifting.

In [1]:
# %pip install "transformers>=5.14,<5.15" torch requests

import json
from pprint import pprint

import requests
import torch
import torch.nn.functional as F

from transformers import (
    GlmOcrConfig,
    GlmOcrForConditionalGeneration,
)

print("torch:", torch.__version__)

/tmp/mm101-build-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.13.0


## 1. The checkpoint JSON is the architecture contract

Do not instantiate the Python defaults and call the result ‘GLM-OCR’. The released decoder is 1536 wide; the class default in Transformers 5.14 is 1024. Read the JSON that accompanies the weights.

In [2]:
CONFIG_URL = "https://huggingface.co/zai-org/GLM-OCR/raw/main/config.json"
released = requests.get(CONFIG_URL, timeout=30).json()

v = released["vision_config"]
t = released["text_config"]
summary = {
    "architecture": released["architectures"][0],
    "vision": {k: v[k] for k in ("depth", "hidden_size", "num_heads", "patch_size", "spatial_merge_size", "out_hidden_size")},
    "text": {k: t[k] for k in ("num_hidden_layers", "hidden_size", "intermediate_size", "num_attention_heads", "num_key_value_heads", "vocab_size")},
    "checkpoint_nextn_layers": t.get("num_nextn_predict_layers"),
}
pprint(summary)

assert v["out_hidden_size"] == t["hidden_size"] == 1536
assert v["spatial_merge_size"] == 2

{'architecture': 'GlmOcrForConditionalGeneration',
 'checkpoint_nextn_layers': 1,
 'text': {'hidden_size': 1536,
          'intermediate_size': 4608,
          'num_attention_heads': 16,
          'num_hidden_layers': 16,
          'num_key_value_heads': 8,
          'vocab_size': 59392},
 'vision': {'depth': 24,
            'hidden_size': 1024,
            'num_heads': 16,
            'out_hidden_size': 1536,
            'patch_size': 14,
            'spatial_merge_size': 2}}


## 2. Build the same topology at toy scale

The tiny config preserves the relationships that matter: vision width → 2×2 strided downsample → text width, GQA in the decoder, and image placeholders inside a decoder-only model. It changes depth and width so the whole graph fits comfortably on a CPU.

In [3]:
tiny_config = GlmOcrConfig(
    vision_config={
        "depth": 2,
        "hidden_size": 64,
        "intermediate_size": 128,
        "num_heads": 4,
        "image_size": 56,
        "patch_size": 14,
        "temporal_patch_size": 2,
        "spatial_merge_size": 2,
        "out_hidden_size": 96,
    },
    text_config={
        "vocab_size": 256,
        "hidden_size": 96,
        "intermediate_size": 192,
        "num_hidden_layers": 2,
        "num_attention_heads": 4,
        "num_key_value_heads": 2,
        "max_position_embeddings": 512,
        "pad_token_id": 0,
        "rope_parameters": {"rope_type": "default", "rope_theta": 10000},
    },
    image_start_token_id=250,
    image_end_token_id=251,
    image_token_id=252,
    video_start_token_id=253,
    video_end_token_id=254,
    video_token_id=255,
)
model = GlmOcrForConditionalGeneration(tiny_config).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"tiny parameters: {n_params:,}")
print(model.model.visual)

tiny parameters: 491,712
GlmOcrVisionModel(
  (patch_embed): GlmOcrVisionPatchEmbed(
    (proj): Conv3d(3, 64, kernel_size=(2, 14, 14), stride=(2, 14, 14))
  )
  (rotary_pos_emb): GlmOcrVisionRotaryEmbedding()
  (blocks): ModuleList(
    (0-1): 2 x GlmOcrVisionBlock(
      (norm1): GlmOcrRMSNorm((64,), eps=1e-05)
      (norm2): GlmOcrRMSNorm((64,), eps=1e-05)
      (attn): GlmOcrVisionAttention(
        (qkv): Linear(in_features=64, out_features=192, bias=True)
        (proj): Linear(in_features=64, out_features=64, bias=True)
        (q_norm): GlmOcrRMSNorm((16,), eps=1e-05)
        (k_norm): GlmOcrRMSNorm((16,), eps=1e-05)
      )
      (mlp): GlmOcrVisionMlp(
        (gate_proj): Linear(in_features=64, out_features=128, bias=True)
        (up_proj): Linear(in_features=64, out_features=128, bias=True)
        (down_proj): Linear(in_features=128, out_features=64, bias=True)
        (act_fn): SiLUActivation()
      )
    )
  )
  (merger): GlmOcrVisionPatchMerger(
    (proj): Linear(in_

## 3. Trace one synthetic image grid

`grid_thw=[1,4,6]` means 24 patches before merging. A 2×2 merge must leave `1×2×3 = 6` visual tokens. `pixel_values` is already flattened into patch payloads by the processor, so each row contains `3×2×14×14` scalars.

In [4]:
activations = {}
handles = []

def save(name):
    def hook(_module, inputs, output):
        activations[name] = {
            "input": inputs[0].detach(),
            "output": output.detach() if isinstance(output, torch.Tensor) else output,
        }
    return hook

vision = model.model.visual
for name, module in {
    "patch_embed": vision.patch_embed,
    "last_block": vision.blocks[-1],
    "post_layernorm": vision.post_layernorm,
    "downsample": vision.downsample,
    "merger": vision.merger,
}.items():
    handles.append(module.register_forward_hook(save(name)))

grid_thw = torch.tensor([[1, 4, 6]], dtype=torch.long)
num_patches = int(grid_thw.prod())
patch_payload = 3 * tiny_config.vision_config.temporal_patch_size * tiny_config.vision_config.patch_size**2
pixel_values = torch.randn(num_patches, patch_payload)

with torch.no_grad():
    vision_output = vision(pixel_values, grid_thw=grid_thw)

for handle in handles:
    handle.remove()

def shape_of(value):
    if isinstance(value, torch.Tensor):
        return tuple(value.shape)
    return type(value).__name__

for name, values in activations.items():
    print(f"{name:14s} {shape_of(values['input'])} -> {shape_of(values['output'])}")

expected_tokens = int(grid_thw[0, 0] * (grid_thw[0, 1] // 2) * (grid_thw[0, 2] // 2))
print("visual prefix:", tuple(vision_output.pooler_output.shape))
assert vision_output.pooler_output.shape == (expected_tokens, 96)

patch_embed    (24, 1176) -> (24, 64)
last_block     (24, 64) -> (24, 64)
post_layernorm (24, 64) -> (24, 64)
downsample     (6, 64, 2, 2) -> (6, 96, 1, 1)
merger         (6, 96) -> (6, 96)
visual prefix: (6, 96)


## 4. Reproduce the 2×2 downsample

The compression is a learned strided convolution, not average pooling and not a lossless pixel shuffle. Recompute it with the functional operator and require exact agreement with the module path.

In [5]:
downsample_input = activations["downsample"]["input"]
library_output = activations["downsample"]["output"]
manual_output = F.conv2d(
    downsample_input,
    vision.downsample.weight,
    vision.downsample.bias,
    stride=vision.downsample.stride,
    padding=vision.downsample.padding,
)
torch.testing.assert_close(manual_output, library_output)
print("assert_close passed:", tuple(manual_output.shape))

assert_close passed: (6, 96, 1, 1)


## 5. Inspect the fusion invariant

The visual tower returned six 96-wide vectors. A valid prompt therefore needs six image-placeholder positions. `get_placeholder_mask` checks that count before any replacement occurs. This is the same invariant chapter 08 found in Gemma 4, reached through a different processor.

In [6]:
input_ids = torch.tensor([[1, 250, *([252] * expected_tokens), 251, 2]])
inputs_embeds = model.get_input_embeddings()(input_ids)
image_mask, _ = model.model.get_placeholder_mask(
    input_ids=input_ids,
    inputs_embeds=inputs_embeds,
    image_features=vision_output.pooler_output,
)
print("placeholder positions:", int(image_mask.sum()))
assert int(image_mask.sum()) == expected_tokens

placeholder positions: 6


## Exercises

1. Sweep `grid_thw` over `(1, 4, 4)`, `(1, 8, 6)`, and `(1, 12, 8)`. Verify that visual tokens grow with image area after the fixed 4× reduction.
2. Replace the learned downsample weights with a hand-constructed average and compare the merger activations. Which operation loses more information in principle?
3. Change `num_key_value_heads` from 2 to 1 and count decoder parameters. Why does the vision tower stay unchanged?
4. Delete one image placeholder and observe the failure. Why is an early shape error preferable to silently truncating features?
5. Open `modeling_glm_ocr.py` and search for `num_nextn_predict_layers`. What does its absence tell you about the `transformers.generate()` path?